Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Load Data

In [3]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time']) 
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df) 

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [4]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

In [5]:
# Create test value
manualX = X_test
manualY = y_test
print(manualY)

[-0.09924674 -0.09546565 -0.12112098 ... -0.01743513 -0.03103252
 -0.04362074]


GBR Model

In [6]:
import onnx

def get_tree_attributes(onnx_file):
    onnx_model =  onnx.load(onnx_file)
    tree_ensemble_node = None
    for node in onnx_model.graph.node:
        if node.op_type == "TreeEnsembleRegressor":
            tree_ensemble_node = node
            break
    
    if tree_ensemble_node is None:
        raise ValueError("No TreeEnsembleRegressor found in the ONNX graph.")
        
    attributes = {}
    # Inspect the model graph's attributes
    for attr in tree_ensemble_node.attribute:
        if attr.name == "nodes_modes":
            attributes[attr.name] = [s.decode('utf-8') for s in attr.strings]
        elif attr.type == onnx.AttributeProto.AttributeType.FLOATS:
            attributes[attr.name] = list(attr.floats)
        elif attr.type == onnx.AttributeProto.AttributeType.INTS:
            attributes[attr.name] = list(attr.ints)
        elif attr.name == "post_transform":
            attributes[attr.name] = attr.s.decode('utf-8')
    return attributes


In [7]:
class TreeNode:
    def __init__(self, node_id, mode, feature_id=None, threshold=None, true_child=None, false_child=None, leaf_value=None):
        self.node_id = node_id
        self.mode = mode
        self.feature_id = feature_id
        self.threshold = threshold
        self.true_child = true_child
        self.false_child = false_child
        self.leaf_value = leaf_value
        
class DecisionTree:
    def __init__(self, tree_id, node_map):
        self.tree_id = tree_id
        self.node_map = node_map
        self.root = self.node_map[0]
    
        
    def predict(self, x):
        """Walk the tree to find the prediction for a single sample."""
        node = self.root
        while node.mode != 'LEAF':
            feature_value = x[node.feature_id]
            
            # Note: handle different node modes (LEQ, LT, etc.)
            if node.mode == 'BRANCH_LEQ':
                if feature_value <= node.threshold:
                    node = node.true_child
                else:
                    node = node.false_child
            # Add other modes as needed
            else:
                raise NotImplementedError(f"Unsupported node mode: {node.mode}")
        
        return node.leaf_value

In [8]:
def reconstruct_gbr_model(attributes):
    """
    Reconstructs the full GBR model from the parsed ONNX attributes.
    """
    all_trees = []
    
    # Group nodes by tree
    nodes_by_tree = {}
    for i, tree_id in enumerate(attributes['nodes_treeids']):
        if tree_id not in nodes_by_tree:
            nodes_by_tree[tree_id] = []
        nodes_by_tree[tree_id].append(i)

        
    # Create a mapping for leaf weights
    leaf_weight_map = {}
    for i, (tree_id, node_id) in enumerate(zip(attributes['target_treeids'], attributes['target_nodeids'])):
        if (tree_id, node_id) not in leaf_weight_map:
            leaf_weight_map[(tree_id, node_id)] = attributes['target_weights'][i]

    # Reconstruct each individual tree
    for tree_id, indices in nodes_by_tree.items():
        node_map = {}
        # Create TreeNode objects
        for i in indices:
            node_id = attributes['nodes_nodeids'][i]
            mode = attributes['nodes_modes'][i]
            
            if mode == 'LEAF':
                leaf_value = leaf_weight_map.get((tree_id, node_id))
                node_map[node_id] = TreeNode(node_id, mode, leaf_value=leaf_value)
            else: # It's a branch node
                feature_id = attributes['nodes_featureids'][i]
                threshold = attributes['nodes_values'][i]
                node_map[node_id] = TreeNode(node_id, mode, feature_id=feature_id, threshold=threshold)
        
        # Link children
        for i in indices:
            node_id = attributes['nodes_nodeids'][i]
            mode = attributes['nodes_modes'][i]
            if mode != 'LEAF':
                true_child_id = attributes['nodes_truenodeids'][i]
                false_child_id = attributes['nodes_falsenodeids'][i]
                node_map[node_id].true_child = node_map[true_child_id]
                node_map[node_id].false_child = node_map[false_child_id]
        
        all_trees.append(DecisionTree(tree_id, node_map))

    return all_trees, attributes['base_values']


In [9]:
# Get the attributes of the GBR model from the onnx file
onnx_file = "ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx"
attributes = get_tree_attributes(onnx_file)

# Reconstruct the trees and get base values
trees, base_values = reconstruct_gbr_model(attributes)

# Use the reconstructed model for prediction
def predict_gbr(x, trees, base_values):
    prediction = base_values[0]
    for tree in trees:
        prediction += tree.predict(x)
    return prediction

In [11]:
# test prediction
final_prediction = []
for item in manualX:
    final_prediction.append(predict_gbr(item, trees, base_values))
print(f"Prediction: {final_prediction}")

Prediction: [-0.14729746644580377, -0.09547406251173829, -0.12677386627477194, -0.1286063865022813, -0.13088162411231696, -0.1226090769787449, -0.11950423063070748, -0.1127720530014189, -0.09571961576061216, -0.08200142897389373, -0.07572274112525879, -0.07197931642469957, -0.07720162547628906, -0.07229110967859942, -0.061477296599872666, -0.053544151739783, -0.0486346362619523, -0.03610569775439476, -0.012144419420892838, -5.400410133038491e-05, -0.01564141834113375, -0.04176210729518548, -0.07923884275528881, -0.06836673238244106, -0.1528587763633995, -0.11005089724760087, -0.16704282746119947, -0.14796576235171433, -0.13764907722777497, -0.11891710400477695, -0.08667074226685845, -0.08947601380218106, -0.08227101501624379, -0.06265742365932869, -0.05124068000297388, -0.03786024046662906, -0.032936563578020106, -0.032092450050712706, -0.02574699450944795, -0.016411271424702, -0.010694028483656304, 0.0001242757662813787, 0.022371388653382063, 0.02728902452226034, 0.01721818829993449, 

In [13]:
inputsL= manualX
#print(inputsL)

import onnxruntime as rt
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx")
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/gbr_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(inputsL, dtype=np.float32) 
#input_data = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)

[array([[-0.14729747],
       [-0.09547409],
       [-0.12677392],
       ...,
       [-0.0155541 ],
       [-0.05449347],
       [-0.03269795]], shape=(70128, 1), dtype=float32)]


In [14]:
from sklearn.metrics import mean_absolute_error
old_out = []
for item in outputs[0]:
    old_out.append(item[0])
print(mean_absolute_error(old_out,final_prediction))

2.443504483734853e-08
